In [0]:
fact_accountsreceivable_recentQuarters=dbutils.widgets.get("fact_accountsreceivable_recentQuarters")
fact_accountsreceivable=dbutils.widgets.get("fact_accountsreceivable")
date=dbutils.widgets.get("date")
fact_accountsreceivable_bkp=dbutils.widgets.get("fact_accountsreceivable_bkp")

In [0]:
spark.sql(f"TRUNCATE TABLE {fact_accountsreceivable_recentQuarters}")


In [0]:
spark.sql(f"""
INSERT INTO {fact_accountsreceivable_recentQuarters}

WITH current_quarter AS (
    SELECT
        DateKey AS WeekEndingDateKey,
        FiscalQuarterNbr
    FROM {date}
    WHERE DateKey = (
        SELECT MAX(reporting_week_ending_date_key)
        FROM {fact_accountsreceivable}
    )
),
prior_quarter AS (
    SELECT
        MIN(DateKey) AS WeekEndingDateKey,
        MIN(FiscalQuarterNbr) AS FiscalQuarterNbr
    FROM {date}
    WHERE FiscalQuarterNbr = (
        SELECT MAX(FiscalQuarterNbr)
        FROM {date}
        WHERE FiscalQuarterNbr < (
            SELECT FiscalQuarterNbr FROM current_quarter
        )
    )
    AND WeekEndingDate = CalendarDate
)

SELECT 
    reporting_week_ending_date_key,
    start_date_key,
    end_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    invoice_balance_type_key,
    age_from_todays_date,
    age_from_quarter_end_date,
    invoice_number,
    balance,
    episode_in_progress
FROM {fact_accountsreceivable}
WHERE reporting_week_ending_date_key >= (
    SELECT WeekEndingDateKey FROM prior_quarter
)
""")